# Evaluate retrieval

## Paths

In [2]:
# find project files

from pathlib import Path


cwd = Path.cwd().resolve()

candidates = []

for path in [cwd, *cwd.parents]:
    candidates.extend([
        path,
        path / "fitness-assistant",
        path / "07-project-example" / "fitness-assistant",
        path / "datatalks" / "llm" / "zoomcamp-llm-2026" / "07-project-example" / "fitness-assistant",
    ])

PROJECT_DIR = None

for candidate in candidates:
    if (candidate / "data" / "data.csv").exists():
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find fitness-assistant/data/data.csv")

COURSE_ROOT = None

for path in [PROJECT_DIR, *PROJECT_DIR.parents]:
    if (path / "pyproject.toml").exists():
        COURSE_ROOT = path
        break

if COURSE_ROOT is None:
    raise FileNotFoundError("Could not find course root with pyproject.toml")

DATA_DIR = PROJECT_DIR / "data"

print("Project dir:", PROJECT_DIR)
print("Data dir:", DATA_DIR)

Project dir: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/07-project-example/fitness-assistant
Data dir: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/07-project-example/fitness-assistant/data


## Packages

In [3]:
# load packages and OpenAI

import json
import os
import random

import pandas as pd
from dotenv import load_dotenv
from minsearch import Index
from openai import OpenAI
from tqdm.auto import tqdm


load_dotenv(COURSE_ROOT / ".env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing. Add it to the course root .env file.")

openai_client = OpenAI()
MODEL = "gpt-5.4-mini"

MODEL

'gpt-5.4-mini'

## Load data

In [4]:
# load the generated exercise dataset

csv_path = DATA_DIR / "data.csv"

df = pd.read_csv(csv_path)
documents = df.to_dict(orient="records")

print("Rows:", len(df))
df.head()

Rows: 50


,id,exercise_name,type_of_activity,type_of_equipment,body_part,type,muscle_groups_activated,instructions
0,push-up-001,Push-Up,Strength,None (bodyweight),Chest,Compound,"Chest, Triceps, Shoulders, Core",Start in a high plank with hands slightly wide...
1,squat-002,Bodyweight Squat,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Core",Stand with feet shoulder-width apart and toes ...
2,plank-003,Plank,Strength,None (bodyweight),Core,Isometric,"Core, Shoulders, Glutes",Place your forearms on the floor with elbows u...
3,lunges-004,Forward Lunge,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Calves",Stand tall with feet hip-width apart. Step for...
4,jumping-jacks-005,Jumping Jacks,Cardio,None (bodyweight),Full Body,Compound,"Shoulders, Quadriceps, Calves, Glutes, Core",Stand upright with feet together and arms at y...


## Build index

In [5]:
# same minsearch index style from the first RAG notebook

index = Index(
    text_fields=[
        "exercise_name",
        "type_of_activity",
        "type_of_equipment",
        "body_part",
        "type",
        "muscle_groups_activated",
        "instructions",
    ],
    keyword_fields=["id"],
)

index.fit(documents)

## Ground truth prompt

In [6]:
# ask the model to create retrieval questions for each exercise record

prompt1_template = """
You are a fitness expert generating evaluation questions.
For the exercise below, generate 2 questions that a user might ask.
Return only a JSON array with objects containing 'id' and 'question' fields.

Exercise: {exercise_name}
Activity type: {type_of_activity}
Equipment: {type_of_equipment}
Body part: {body_part}
Muscle groups: {muscle_groups_activated}
Instructions: {instructions}
""".strip()

## Generate questions

In [7]:
# this calls OpenAI, so it reuses the saved file if it already exists

questions_path = DATA_DIR / "ground-truth-retrieval.csv"

if questions_path.exists():
    df_questions = pd.read_csv(questions_path)
else:
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        prompt = prompt1_template.format(**row.to_dict())

        response = openai_client.responses.create(
            model=MODEL,
            input=[{"role": "user", "content": prompt}],
        )

        questions = json.loads(response.output_text)

        for q in questions:
            q["id"] = row["id"]
            results.append(q)

    df_questions = pd.DataFrame(results)
    df_questions.to_csv(questions_path, index=False)

print("Questions:", len(df_questions))
df_questions.head()

  0%|          | 0/50 [00:00<?, ?it/s]

Questions: 100


,id,question
0,push-up-001,How do I perform a push-up with proper form?
1,push-up-001,Which muscles does a push-up work?
2,squat-002,How can I make sure my knees stay aligned over...
3,squat-002,How low should I go in a bodyweight squat whil...
4,plank-003,How long should I hold a plank as a beginner?


## Load ground truth

In [8]:
# each question has the id of the exercise it should retrieve

df_questions = pd.read_csv(DATA_DIR / "ground-truth-retrieval.csv")
ground_truth = df_questions.to_dict(orient="records")

print("Ground truth questions:", len(ground_truth))
ground_truth[:3]

Ground truth questions: 100


[{'id': 'push-up-001',
  'question': 'How do I perform a push-up with proper form?'},
 {'id': 'push-up-001', 'question': 'Which muscles does a push-up work?'},
 {'id': 'squat-002',
  'question': 'How can I make sure my knees stay aligned over my toes during bodyweight squats?'}]

## Metrics

In [9]:
# Hit Rate checks whether the correct document appears anywhere in the results


def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt += 1

    return cnt / len(relevance_total)


# MRR gives more score when the correct document appears higher in the ranking


def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank]:
                total_score += 1 / (rank + 1)
                break

    return total_score / len(relevance_total)


# run a search function and compare returned ids with the expected id


def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q["id"]
        results = search_function(q)
        relevance = [d["id"] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## Search function

In [10]:
# baseline search uses no boost unless we pass boost values


def minsearch_search(query, boost=None):
    if boost is None:
        boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10,
    )

    return results

## Baseline

In [11]:
# first check retrieval quality before tuning

baseline_metrics = evaluate(
    ground_truth,
    lambda q: minsearch_search(q["question"]),
)

baseline_metrics

  0%|          | 0/100 [00:00<?, ?it/s]

{'hit_rate': 1.0, 'mrr': 0.9403333333333335}

## Split data

In [12]:
# use validation for tuning and test for final checking

split_idx = int(len(df_questions) * 0.7)

if split_idx == 0 or split_idx == len(df_questions):
    raise ValueError("Not enough ground truth rows for validation/test split")

df_validation = df_questions[:split_idx]
df_test = df_questions[split_idx:]

gt_val = df_validation.to_dict(orient="records")
gt_test = df_test.to_dict(orient="records")

print("Validation:", len(gt_val))
print("Test:", len(gt_test))

Validation: 70
Test: 30


## Tune boosts

In [13]:
# random search tries different boost values and keeps the best one

random.seed(1)


def simple_optimize(param_ranges, objective_function, n_iterations=20):
    best_params = None
    best_score = float("-inf")

    for _ in range(n_iterations):
        current_params = {}

        for field, (low, high) in param_ranges.items():
            current_params[field] = random.uniform(low, high)

        current_score = objective_function(current_params)

        if current_score > best_score:
            best_score = current_score
            best_params = current_params

    return best_params


param_ranges = {
    "exercise_name": (0.0, 3.0),
    "type_of_activity": (0.0, 3.0),
    "type_of_equipment": (0.0, 3.0),
    "body_part": (0.0, 3.0),
    "type": (0.0, 3.0),
    "muscle_groups_activated": (0.0, 3.0),
    "instructions": (0.0, 3.0),
}


def objective(boost_params):
    def search_function(q):
        return minsearch_search(q["question"], boost=boost_params)

    results = evaluate(gt_val, search_function)
    return results["hit_rate"]


best_params = simple_optimize(param_ranges, objective, n_iterations=20)

best_params

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

...

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

{'exercise_name': 0.40309273233720366,
 'type_of_activity': 2.542301210811698,
 'type_of_equipment': 2.291323856929842,
 'body_part': 0.7652070772182651,
 'type': 1.486305261275823,
 'muscle_groups_activated': 1.3484731943662145,
 'instructions': 1.9547789181682889}

## Validation score

In [14]:
# check the tuned boost values on validation data

validation_metrics = evaluate(
    gt_val,
    lambda q: minsearch_search(q["question"], boost=best_params),
)

validation_metrics

  0%|          | 0/70 [00:00<?, ?it/s]

{'hit_rate': 1.0, 'mrr': 0.7240816326530612}

## Test score

In [15]:
# final check on test data that was not used for tuning

test_metrics = evaluate(
    gt_test,
    lambda q: minsearch_search(q["question"], boost=best_params),
)

test_metrics

  0%|          | 0/30 [00:00<?, ?it/s]

{'hit_rate': 1.0, 'mrr': 0.9027777777777778}

## Save results

In [16]:
# save the metrics so the next notebook can reference the retrieval result

results = [
    {"run": "baseline", **baseline_metrics},
    {"run": "tuned_validation", **validation_metrics},
    {"run": "tuned_test", **test_metrics},
]

df_results = pd.DataFrame(results)
df_results.to_csv(DATA_DIR / "retrieval-evaluation-results.csv", index=False)

df_results

,run,hit_rate,mrr
0,baseline,1.0,0.940333
1,tuned_validation,1.0,0.724082
2,tuned_test,1.0,0.902778


## Final search

In [17]:
# use this search function in the next RAG evaluation notebook


def search(query):
    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=best_params,
        num_results=10,
    )

    return results


search("Give me leg exercises for hamstrings")[:3]

[{'id': 'single-leg-romanian-deadlift-046',
  'exercise_name': 'Single-Leg Romanian Deadlift',
  'type_of_activity': 'Strength',
  'type_of_equipment': 'Dumbbells',
  'body_part': 'Hamstrings',
  'type': 'Compound',
  'muscle_groups_activated': 'Hamstrings, Glutes, Core, Balance Muscles',
  'instructions': 'Stand on one leg holding dumbbells or a single weight. Hinge at the hips while extending the free leg behind you and lowering the weight toward the floor. Keep your back flat and hips square. Return to standing by driving through the standing heel and squeezing the glute.'},
 {'id': 'hamstring-curl-021',
  'exercise_name': 'Lying Hamstring Curl',
  'type_of_activity': 'Strength',
  'type_of_equipment': 'Machine',
  'body_part': 'Hamstrings',
  'type': 'Isolation',
  'muscle_groups_activated': 'Hamstrings, Calves',
  'instructions': 'Lie face down on the hamstring curl machine and position the pad just above your ankles. Bend your knees to curl the pad toward your glutes. Pause brief